In [1]:
import pandas as pd
import numpy as np

In [2]:
vendedores_ativos_geral = ['Bryan Casarotto',
'Daniel Nunes de Paula Junior',
'Daniele Schmitz',
'GILBERTO LIMA DE PINHO JUNIOR',
'Kelli de Almeida Ferreira',
'KELLI DE ALMEIDA FERREIRA',
'Laura Vitoria Da Silveira Trindade',
'Leonardo Bianchi',
'Letícia Eduarda Cruz',
'LUCAS VASCONCELOS BATTAGLIA KRAUSE',
'RONALDO DA COSTA BARRIOS',
'Ruan da Silva',
'TALIA LINS RAMOS',
'Willian Luiz Pereira',
'Amanda Dias do Amaral',
'CRISTIAN RHEINHEIMER',
'Eduardo Dutra de Lima',
'GUSTAVO BALBINOTT VENASSI',
'Guilherme Rafael Hartmann Soares',
'Gustavo Cesar Burnier',
'JOAO PEDRO MOCELIN',
'Joao Gustavo Santian Da Silva',
'Kaylane Victoria Sousa Sa',
'LUCAS SIMOES BERNART',
'MARJANA  KUHN',
'MAURICIO HENRIQUE CESCO',
'MURILO DUARTE DA SILVA',
'Sidinei Da Silva Dias',
'TIAGO PEDROSO DA SILVA'
]

vendedores_ativos_helder = ['Bryan Casarotto',
'Daniel Nunes de Paula Junior',
'Daniele Schmitz',
'GILBERTO LIMA DE PINHO JUNIOR',
'Kelli de Almeida Ferreira',
'Laura Vitoria Da Silveira Trindade',
'Leonardo Bianchi',
'Letícia Eduarda Cruz',
'LUCAS VASCONCELOS BATTAGLIA KRAUSE',
'RONALDO DA COSTA BARRIOS',
'Ruan da Silva',
'TALIA LINS RAMOS',
'Willian Luiz Pereira'
]

vendedores_ativos_karen =['Amanda Dias do Amaral',
'CRISTIAN RHEINHEIMER',
'Eduardo Dutra de Lima',
'Gabriel Henrique Zuanazzi',
'GUSTAVO BALBINOTT VENASSI',
'Guilherme Rafael Hartmann Soares',
'Gustavo Cesar Burnier',
'JOAO PEDRO MOCELIN',
'Joao Gustavo Santian Da Silva',
'Kaylane Victoria Sousa Sa',
'LUCAS SIMOES BERNART',
'MARJANA  KUHN',
'MAURICIO HENRIQUE CESCO',
'MURILO DUARTE DA SILVA',
'Sidinei Da Silva Dias',
'TIAGO PEDROSO DA SILVA'
]

#### Parte Helder

In [3]:
data = {'Nome_Vendedor': vendedores_ativos_helder}
df_vendedores_helder = pd.DataFrame(data)
df_vendedores_helder

,Nome_Vendedor
0,Bryan Casarotto
1,Daniel Nunes de Paula Junior
2,Daniele Schmitz
3,GILBERTO LIMA DE PINHO JUNIOR
4,Kelli de Almeida Ferreira
5,Laura Vitoria Da Silveira Trindade
6,Leonardo Bianchi
7,Letícia Eduarda Cruz
8,LUCAS VASCONCELOS BATTAGLIA KRAUSE
9,RONALDO DA COSTA BARRIOS


In [4]:
df_churn_revenda = pd.read_excel('./data/churn_por_class_5_7.xlsx', sheet_name='Sheet1')
df_churn_revenda = df_churn_revenda[['Raiz_CNPJ', 'Conta_ID', 'tipo_conta', 'Razao_Social_Pessoas', 'CNPJ', 'Grupo_Econômico_ID', 'Grupo_Econômico_Nome', 'Classificacao_Pessoa', 'Categoria_Porte']].drop_duplicates(subset='Raiz_CNPJ')
df_churn_revenda

,Raiz_CNPJ,Conta_ID,tipo_conta,Razao_Social_Pessoas,CNPJ,Grupo_Econômico_ID,Grupo_Econômico_Nome,Classificacao_Pessoa,Categoria_Porte
0,13462,135019,2,SOFTSYS INFORMATICA E SEGURANCA ELETRONICA LTDA,13462000143,NaN,NaN,7,Micro
1,56500,79335,2,MICROWORK COMERCIO E ASSISTENCIA TECNICA LTDA,56500000145,NaN,NaN,7,Desconhecido
2,99715,73770,2,BIT LIFE INFORMATICA LTDA,99715000143,NaN,NaN,5,Desconhecido
3,168556,68930,2,S B C Informatica Ltda,168556000191,NaN,NaN,5,Micro
4,168569,130921,2,CLAITOM SEGA E CIA LTDA,168569000160,NaN,NaN,5,Desconhecido
...,...,...,...,...,...,...,...,...,...
1259,89829667,92616,2,FIGHERA MAQUINAS PARA ESCRITORIO LTDA,89829667000105,NaN,NaN,7,Micro
1260,91445577,141914,2,SERGIO GIRARDI,91445577000162,NaN,NaN,5,Micro
1261,91981027,131697,2,BM ELETRO ELETRONICA LTDA,91981027000168,NaN,NaN,7,Desconhecido
1262,93111441,91442,2,UNITEC INFORMATICA LTDA,93111441000141,NaN,NaN,5,Micro


In [5]:
def distribuir_clientes_helder(vendedores, clientes, clientes_por_vendedor=50):
    num_vendedores = len(vendedores)

    num_clientes = len(clientes)

    if num_clientes < clientes_por_vendedor * num_vendedores:
        st.warning(f'Temos apenas {num_clientes} clientes e não conseguimos distribuir {clientes_por_vendedor} por vendedor.')
        clientes_por_vendedor = num_clientes // num_vendedores

    clientes_distribuidos = np.array_split(clientes['Razao_Social_Pessoas'], num_vendedores)

    tabela_distribuicao = []
    for i, vendedor in enumerate(vendedores['Nome_Vendedor']):
        clientes_vendedor = clientes_distribuidos[i][:clientes_por_vendedor]
        for cliente in clientes_vendedor:
            tabela_distribuicao.append([vendedor, cliente])

    tabela_distribuicao = pd.DataFrame(tabela_distribuicao, columns=['Vendedor', 'Cliente'])

    clientes_distribuidos_ids = pd.concat([df_churn_revenda[df_churn_revenda['Razao_Social_Pessoas'] == cliente] for cliente in tabela_distribuicao['Cliente']])['Raiz_CNPJ'].unique()
    clientes_sobrando = clientes[~clientes['Raiz_CNPJ'].isin(clientes_distribuidos_ids)]

    return tabela_distribuicao, clientes_sobrando

In [6]:
def to_excel(df):
    output = BytesIO()
    with pd.ExcelWriter(output, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, sheet_name='Clientes Distribuídos')
    return output.getvalue()

In [10]:
def mostrar_clientes(vendedor_selecionado, tabela_distribuicao):
    clientes_vendedor = tabela_distribuicao[tabela_distribuicao['Vendedor'] == vendedor_selecionado]
    print(f'Clientes distribuidos para {vendedor_selecionado}')
    clientes_vendedor
    excel_data = to_excel(clientes_vendedor)

## Distribuição de Clientes para Vendedores
#### Vendedores

In [7]:
df_vendedores_helder

,Nome_Vendedor
0,Bryan Casarotto
1,Daniel Nunes de Paula Junior
2,Daniele Schmitz
3,GILBERTO LIMA DE PINHO JUNIOR
4,Kelli de Almeida Ferreira
5,Laura Vitoria Da Silveira Trindade
6,Leonardo Bianchi
7,Letícia Eduarda Cruz
8,LUCAS VASCONCELOS BATTAGLIA KRAUSE
9,RONALDO DA COSTA BARRIOS


#### Clientes Cadastrados

In [8]:
df_churn_revenda

,Raiz_CNPJ,Conta_ID,tipo_conta,Razao_Social_Pessoas,CNPJ,Grupo_Econômico_ID,Grupo_Econômico_Nome,Classificacao_Pessoa,Categoria_Porte
0,13462,135019,2,SOFTSYS INFORMATICA E SEGURANCA ELETRONICA LTDA,13462000143,NaN,NaN,7,Micro
1,56500,79335,2,MICROWORK COMERCIO E ASSISTENCIA TECNICA LTDA,56500000145,NaN,NaN,7,Desconhecido
2,99715,73770,2,BIT LIFE INFORMATICA LTDA,99715000143,NaN,NaN,5,Desconhecido
3,168556,68930,2,S B C Informatica Ltda,168556000191,NaN,NaN,5,Micro
4,168569,130921,2,CLAITOM SEGA E CIA LTDA,168569000160,NaN,NaN,5,Desconhecido
...,...,...,...,...,...,...,...,...,...
1259,89829667,92616,2,FIGHERA MAQUINAS PARA ESCRITORIO LTDA,89829667000105,NaN,NaN,7,Micro
1260,91445577,141914,2,SERGIO GIRARDI,91445577000162,NaN,NaN,5,Micro
1261,91981027,131697,2,BM ELETRO ELETRONICA LTDA,91981027000168,NaN,NaN,7,Desconhecido
1262,93111441,91442,2,UNITEC INFORMATICA LTDA,93111441000141,NaN,NaN,5,Micro


#### Parte Karen

In [17]:
data = {'Nome_Vendedor': vendedores_ativos_karen}
df_vendedores_karen = pd.DataFrame(data)
df_vendedores_karen

,Nome_Vendedor
0,Amanda Dias do Amaral
1,CRISTIAN RHEINHEIMER
2,Eduardo Dutra de Lima
3,Gabriel Henrique Zuanazzi
4,GUSTAVO BALBINOTT VENASSI
5,Guilherme Rafael Hartmann Soares
6,Gustavo Cesar Burnier
7,JOAO PEDRO MOCELIN
8,Joao Gustavo Santian Da Silva
9,Kaylane Victoria Sousa Sa


In [23]:
df_churn_corporativo = pd.read_excel('./data/churn_por_class_restante.xlsx', sheet_name='Sheet1')
df_churn_corporativo = df_churn_corporativo[['Raiz_CNPJ', 'Conta_ID', 'tipo_conta', 'Razao_Social_Pessoas', 'CNPJ', 'Grupo_Econômico_ID', 'Grupo_Econômico_Nome', 'Classificacao_Pessoa', 'Categoria_Porte']].drop_duplicates(subset='Raiz_CNPJ')
df_churn_corporativo

,Raiz_CNPJ,Conta_ID,tipo_conta,Razao_Social_Pessoas,CNPJ,Grupo_Econômico_ID,Grupo_Econômico_Nome,Classificacao_Pessoa,Categoria_Porte
0,00082647,13135,2,Grafica E Editora Blumen Eireli,00082647000100,NaN,NaN,3,Desconhecido
1,00083682,185257,2,ECO RAPIDO TRANSPORTES LTDA,00083682000577,NaN,NaN,2,Micro
2,00102436,123869,2,Condominio Civil Pro-Indiviso Do Shopping Cent...,00102436000191,NaN,NaN,4,Micro
3,00103956,186333,2,UNIMED LITORAL SUL/RS - COOP MEDICA LTDA,00103956000119,NaN,NaN,4,Micro
4,00108578,77823,2,AMAZONIA MAQUINAS E IMPLEMENTOS LTDA,00108578001308,46.0,Grupo Amazonia Maquinas e Implementos LTDA,2,Desconhecido
...,...,...,...,...,...,...,...,...,...
3510,96704333,48250,2,FUNDACAO ARAUCARIA,96704333000413,NaN,NaN,4,Desconhecido
3511,97117386,131432,2,ACUNHA SOLE-ENGENHARIA LTDA,97117386000158,NaN,NaN,4,Micro
3512,97467856,171197,2,RODOPARANA IMPLEMENTOS RODOVIARIOS LTDA,97467856000103,NaN,NaN,2,Micro
3513,97525918,182527,2,SERALLTEC TELECOMUNICACOES E INFORMATICA LTDA,97525918000196,NaN,NaN,4,Desconhecido
